# 🎬 Storyboard Generator — SDXL + ControlNet + IP-Adapter

**Team Notebook — Colab Pro (T4 GPU)**

This notebook implements a simple storyboard generation pipeline using three core technologies:
- **SDXL** — base text-to-image model
- **ControlNet** — enforces spatial layout via depth maps (MiDaS)
- **IP-Adapter** — injects a character reference image to keep faces consistent

---
### 🧑‍💻 Team Roles for Today
| Role | Task |
|---|---|
| **Prompt Engineers** | Edit the `scene_json` in Section 4 and tune the prompt text |
| **Parameter Tuners** | Adjust `ip_adapter_scale` (0.3–0.7) in Section 5 |
| **ControlNet Tuners** | Adjust `controlnet_scale` (0.5–0.8) in Section 5 |

> ⏱️ **Compute tip:** Avoid re-running Sections 1–2 unless necessary — model loading is the most expensive step. Keep `num_inference_steps` at 25 or below during testing.

## Section 1 — Install Dependencies + Verify GPU
Takes ~5 mins (run once per session). Run this first. You should see a CUDA device (e.g. Tesla T4). If not, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
%%capture
!pip install -q --force-reinstall \
  "numpy<2.0.0" \
  "torch==2.3.0+cu121" \
  "torchvision==0.18.0+cu121" \
  --index-url https://download.pytorch.org/whl/cu121

!pip install -q \
  "diffusers==0.30.3" \
  "huggingface_hub>=0.25.0" \
  "transformers==4.44.0" \
  "Pillow<12.0.0" \
  accelerate safetensors controlnet-aux timm matplotlib

In [ ]:
%pip list

## IMPORTANT NOTE
### Once the above section is run, go to three dots next to the off/factory reset button and click "Restart and Clear Cell Outputs". 
We do this so that we can refresh Kaggle's kernel to use all of the packages we just installed; not doing so causes import errors when initializing the models.
### Once restarted, you don't have to rerun the above cells; continue from the next cell.

In [ ]:
import torch
import torchvision

print("Torch:", torch.__version__)
print("Vision:", torchvision.__version__)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected — go to Runtime → Change runtime type → GPU (T4)")

## Section 2 — Load Models
> ⏱️ Takes ~4–6 minutes on first run (downloads weights). Subsequent runs in the same session are instant.

We load:
1. **SDXL base** (text-to-image backbone)
2. **ControlNet** conditioned on depth maps
3. **IP-Adapter** for character reference injection
4. **MiDaS** depth estimator

In [ ]:
import torch
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    AutoencoderKL,
)
from diffusers.utils import load_image
from transformers import pipeline as hf_pipeline
from PIL import Image
import numpy as np
import requests
from io import BytesIO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32

print("Loading ControlNet (depth)...")
controlnet = ControlNetModel.from_pretrained(
    "diffusers/controlnet-depth-sdxl-1.0",
    torch_dtype=DTYPE,
    use_safetensors=True,
)

print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(
    "madebyollin/sdxl-vae-fp16-fix",
    torch_dtype=DTYPE,
)

print("Loading SDXL pipeline...")
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    controlnet=controlnet,
    vae=vae,
    torch_dtype=DTYPE,
    use_safetensors=True,
)

print("Loading IP-Adapter (SDXL)...")
pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="sdxl_models",
    weight_name="ip-adapter_sdxl.bin",
)

# Keep this to save VRAM
pipe.enable_model_cpu_offload()
pipe.enable_vae_tiling()

print("Loading MiDaS depth estimator...")
depth_estimator = hf_pipeline(
    "depth-estimation",
    model="Intel/dpt-large",
    device=0 if DEVICE == "cuda" else -1,
)

print("\n✅ All models loaded!")

## Section 3 — Helper Functions
Utility functions for depth map generation and image display. Just run this cell.

In [ ]:
import matplotlib.pyplot as plt

def get_depth_map(image: Image.Image) -> Image.Image:
    """Run MiDaS on an image and return a depth map as a PIL Image."""
    image = image.convert("RGB")
    depth_out = depth_estimator(image)
    depth_np  = np.array(depth_out["depth"])
    # Normalize to 0-255
    depth_np  = (depth_np - depth_np.min()) / (depth_np.max() - depth_np.min() + 1e-8)
    depth_np  = (depth_np * 255).astype(np.uint8)
    # ControlNet expects a 3-channel image
    depth_img = Image.fromarray(depth_np).convert("RGB")
    depth_img = depth_img.resize(image.size)
    return depth_img


def build_prompt_from_scene(scene: dict) -> str:
    """Convert a scene JSON dict into a rich SDXL prompt string."""
    parts = [
        scene.get("action_note", ""),
        f"{scene.get('shot_type', '')} shot",
        f"{scene.get('camera_angle', '')} angle",
        f"background: {scene.get('background', '')}",
        f"lighting: {scene.get('lighting_mood', '')}",
    ]
    chars = scene.get("characters", [])
    for c in chars:
        parts.append(f"{c.get('name','character')} {c.get('position','')}")
    prompt = ", ".join(p for p in parts if p.strip(", "))
    prompt += ", cinematic storyboard panel, highly detailed"
    return prompt


def show_panels(images: list, titles: list = None):
    """Display a list of PIL images in a grid."""
    n = len(images)
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    axes = np.array(axes).flatten()
    for i, img in enumerate(images):
        axes[i].imshow(img)
        axes[i].axis("off")
        if titles and i < len(titles):
            axes[i].set_title(titles[i], fontsize=9)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    plt.tight_layout()
    plt.show()

print("✅ Helper functions ready")

## Section 4 — Define Your Scene (Prompt Engineers ✏️)

Edit the `SCENE_PANELS` list below. Each dict is one storyboard panel and must match the agreed JSON schema:

```
shot_type      | "close-up" | "medium" | "wide" | "extreme wide"
camera_angle   | "eye-level" | "low angle" | "high angle" | "dutch tilt"
characters     | list of {name, position}  e.g. {"name": "hero", "position": "left foreground"}
lighting_mood  | "golden hour" | "noir" | "overcast" | "dramatic side-light" | ...
background     | plain text description
action_note    | what is happening in the panel
```

You can also upload your own **character reference image** below (used by IP-Adapter).

In [ ]:
# ── PROMPT ENGINEERS: Edit this list ─────────────────────────────────────────

SCENE_PANELS = [
    {
        "shot_type":     "wide",
        "camera_angle":  "eye-level",
        "characters":    [{"name": "hero", "position": "center"}],
        "lighting_mood": "golden hour",
        "background":    "abandoned city street, crumbling skyscrapers",
        "action_note":   "hero stands alone surveying the ruins",
    },
    {
        "shot_type":     "close-up",
        "camera_angle":  "low angle",
        "characters":    [{"name": "hero", "position": "frame-filling"}],
        "lighting_mood": "dramatic side-light",
        "background":    "blurred debris",
        "action_note":   "hero's determined face, clenched jaw",
    },
    {
        "shot_type":     "medium",
        "camera_angle":  "high angle",
        "characters":    [
            {"name": "hero",   "position": "left"},
            {"name": "villain","position": "right"},
        ],
        "lighting_mood": "cold blue overcast",
        "background":    "rooftop, stormy sky",
        "action_note":   "hero and villain face off",
    },
    {
        "shot_type":     "extreme wide",
        "camera_angle":  "eye-level",
        "characters":    [],
        "lighting_mood": "sunset orange",
        "background":    "city skyline silhouette",
        "action_note":   "establishing shot of the city at dusk",
    },
]

# ── CHARACTER REFERENCE IMAGE ─────────────────────────────────────────────────
# Option A: upload a file via Colab's file browser (Files tab on the left)
#           then set CHARACTER_REF_PATH to its path.
# Option B: use a URL (uncomment the second block).
# If neither is set, IP-Adapter is skipped.

CHARACTER_REF_PATH = "/kaggle/input/datasets/sharabhojha/storyboard/storyboard stuff/elmo.png"          # e.g. "/content/my_character.png"
# CHARACTER_REF_URL = "https://..." # uncomment to use a URL instead

# ── LAYOUT REFERENCE IMAGE ───────────────────────────────────────────────────
# Optional rough sketch / layout image for ControlNet depth conditioning.
# If None, a flat grey placeholder is used (ControlNet still runs but has no
# spatial guidance — good for testing ControlNet scale = 0).

LAYOUT_REF_PATHS = [
    "/kaggle/input/datasets/sharabhojha/storyboard/storyboard stuff/Page1.jpg",   # Panel 1 — replace with path/URL or leave None
    "/kaggle/input/datasets/sharabhojha/storyboard/storyboard stuff/Page2.jpg",   # Panel 2
    "/kaggle/input/datasets/sharabhojha/storyboard/storyboard stuff/Page3.jpg",   # Panel 3
    "/kaggle/input/datasets/sharabhojha/storyboard/storyboard stuff/Page4.jpg",   # Panel 4
]             # e.g. "/content/my_layout.png"

# ─────────────────────────────────────────────────────────────────────────────

# Print all prompts so Prompt Engineers can sanity-check them
print("📋 Generated prompts:\n")
for i, scene in enumerate(SCENE_PANELS):
    p = build_prompt_from_scene(scene)
    print(f"Panel {i+1}: {p}\n")

## Section 5 — Tuning Parameters 🎛️

**Parameter Tuners:** Change `ip_adapter_scale`  
**ControlNet Tuners:** Change `controlnet_scale`  

| Parameter | Range | Effect |
|---|---|---|
| `ip_adapter_scale` | 0.3 – 0.7 | Low = follows prompt more; High = sticks to reference face more |
| `controlnet_scale` | 0.5 – 0.8 | Low = looser spatial layout; High = depth map dominates composition |
| `num_inference_steps` | 20 – 40 | More steps = better quality, more compute. **Keep at 25 for testing!** |
| `guidance_scale` | 5 – 12 | How strictly the model follows the prompt |

> 💡 **Log your experiments!** Fill in the table at the bottom of the notebook.

In [ ]:
# ── PARAMETER TUNERS: Edit these ─────────────────────────────────────────────

ip_adapter_scale   = 0.8    # float 0.3 – 0.7
controlnet_scale   = 0.3   # float 0.5 – 0.8
num_inference_steps = 25    # keep ≤ 25 during testing to save compute
guidance_scale     = 7.5    # classifier-free guidance
image_size         = (1024, 1024)  # (width, height)

# Negative prompt — things to avoid in all panels
NEGATIVE_PROMPT = (
    "blurry, low quality, distorted face, extra limbs, watermark, "
    "text, duplicate, bad anatomy, ugly"
)

# ─────────────────────────────────────────────────────────────────────────────
print(f"ip_adapter_scale  = {ip_adapter_scale}")
print(f"controlnet_scale  = {controlnet_scale}")
print(f"inference steps   = {num_inference_steps}")
print(f"guidance scale    = {guidance_scale}")
print(f"image size        = {image_size}")

## Section 6 — Load Reference Images

In [ ]:
W, H = image_size

# ── Character reference (IP-Adapter) ─────────────────────────────────────────
character_ref = None

if CHARACTER_REF_PATH:
    character_ref = Image.open(CHARACTER_REF_PATH).convert("RGB")
    print(f"✅ Loaded character ref from file: {CHARACTER_REF_PATH}")
elif 'CHARACTER_REF_URL' in dir() and CHARACTER_REF_URL:
    resp = requests.get(CHARACTER_REF_URL)
    character_ref = Image.open(BytesIO(resp.content)).convert("RGB")
    print(f"✅ Loaded character ref from URL")
else:
    print("ℹ️  No character reference provided — IP-Adapter will be skipped.")

"""
# ── Layout reference (ControlNet / MiDaS) ────────────────────────────────────
if LAYOUT_REF_PATH:
    layout_img = Image.open(LAYOUT_REF_PATH).convert("RGB").resize((W, H))
    print(f"✅ Loaded layout ref from: {LAYOUT_REF_PATH}")
else:
    # Flat mid-grey placeholder — ControlNet still activates but has no content
    layout_img = Image.fromarray(np.full((H, W, 3), 128, dtype=np.uint8))
    print("ℹ️  No layout ref provided — using grey placeholder for depth input.")
"""

layout_imgs = [Image.open(path).convert("RGB").resize((W, H)) 
               for path in LAYOUT_REF_PATHS]

print("\nGenerating depth map from layout ref...")
depth_maps = [get_depth_map(layout_img) for layout_img in layout_imgs]

# Preview
previews = [item for i in range(0, 4) for item in (layout_imgs[i], depth_maps[i])]
preview_titles = ["Layout Input", "Depth Map (MiDaS)"] * 4
if character_ref:
    previews.insert(0, character_ref)
    preview_titles.insert(0, "Character Reference")

show_panels(previews, preview_titles)

## Section 7 — Generate Storyboard Panels 🚀

> ⏱️ Each panel takes ~25–40 seconds on a T4. 4 panels ≈ 2–3 minutes total.

In [ ]:
import time

# Apply IP-Adapter scale (or disable if no reference)
if character_ref is not None:
    pipe.set_ip_adapter_scale(ip_adapter_scale)
    ip_adapter_image = character_ref
else:
    pipe.set_ip_adapter_scale(0.0)  # disable
    ip_adapter_image = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))

generated_panels = []
panel_titles     = []

for i, scene in enumerate(SCENE_PANELS):
    prompt = build_prompt_from_scene(scene)
    print(f"\n🎨 Generating panel {i+1}/{len(SCENE_PANELS)}...")
    print(f"   Prompt: {prompt[:80]}...")

    t0 = time.time()
    result = pipe(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        image=depth_maps[i],                         # ControlNet input
        ip_adapter_image=ip_adapter_image,        # IP-Adapter input
        controlnet_conditioning_scale=controlnet_scale,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        width=W,
        height=H,
        generator=torch.manual_seed(42 + i),      # reproducible per panel
    )
    elapsed = time.time() - t0

    img = result.images[0]
    generated_panels.append(img)
    title = f"Panel {i+1}: {scene['shot_type']} / {scene['camera_angle']}"
    panel_titles.append(title)
    print(f"   ✅ Done in {elapsed:.1f}s")

print("\n🎬 All panels generated!")
show_panels(generated_panels, panel_titles)

## Section 8 — Save Outputs

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/storyboard_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for i, img in enumerate(generated_panels):
    fname = f"{OUTPUT_DIR}/panel_{i+1:02d}_ip{ip_adapter_scale}_cn{controlnet_scale}.png"
    img.save(fname)
    print(f"Saved: {fname}")

print(f"\n✅ {len(generated_panels)} panels saved to {OUTPUT_DIR}")
print("Download them from the Files tab (left sidebar) → right-click → Download.")

## Section 9 — Experiment Log (Fill This In! 📝)

After each run, copy your settings and observations here so the team can compare results.

| Run | `ip_adapter_scale` | `controlnet_scale` | `steps` | Observations |
|-----|--------------------|--------------------|---------|---------------|
| 1   | 0.5                | 0.65               | 25      | *(baseline)*  |
| 2   |                    |                    |         |               |
| 3   |                    |                    |         |               |
| 4   |                    |                    |         |               |

**Questions to answer:**
- At what `ip_adapter_scale` does the character's face start looking different from the reference?
- At what `controlnet_scale` does the depth map feel too constraining vs. too loose?
- Which panel prompt gave the best result, and why?

---
*Happy generating! 🎬*